In [ ]:
from google.colab import files
uploaded = files.upload()
# A file picker will appear — upload: train.csv, test.csv, sample_submission.csv

Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


In [ ]:
!pip install xgboost lightgbm catboost scikit-learn imbalanced-learn pandas numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00


In [ ]:
"""
=============================================================================
Tata Steel AI Hackathon — Defect Detection in Hot Rolling
=============================================================================
Goal   : Predict Alpha defect (Y=1) with Recall=100%, Precision > 90%
Author : Solution Script — run top-to-bottom to generate submission.csv
=============================================================================

HOW TO RUN (Step-by-Step):
---------------------------
1. Install dependencies:
       pip install xgboost lightgbm catboost scikit-learn imbalanced-learn
                   pandas numpy optuna

2. Place these files in the SAME folder as this script:
       train.csv, test.csv, sample_submission.csv

3. Run:
       python tata_steel_defect_solution.py

4. Output:  submission.csv  ← upload this to HackerEarth
=============================================================================
"""

import os, warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (recall_score, precision_score,
                              f1_score, roc_auc_score, classification_report)

from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# 0.  PATHS  (edit if files are elsewhere)
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR  = "/content"          # folder containing train/test/sample_submission CSVs
OUT_DIR   = "/content"       # where submission.csv is written
CAT_DIR   = "/content/catboost_info"  # catboost working dir
os.makedirs(CAT_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
print("="*60)
print("  STEP 1: Loading data")
print("="*60)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"  Train : {train.shape}  |  Defects: {int(train['Y'].sum())} / {len(train)}"
      f"  ({train['Y'].mean()*100:.1f}%)")
print(f"  Test  : {test.shape}")

FEATURE_COLS = [c for c in train.columns if c not in ["CoilID", "Y"]]
X     = train[FEATURE_COLS].copy()
y     = train["Y"].values.astype(int)
Xtest = test[FEATURE_COLS].copy()

# ─────────────────────────────────────────────────────────────────────────────
# 2.  IMPUTATION  (median per column, fit on train only)
# ─────────────────────────────────────────────────────────────────────────────
print("\n  STEP 2: Imputing missing values")
medians = X.median()
X     = X.fillna(medians)
Xtest = Xtest.fillna(medians)
print(f"  Missing after impute — train: {X.isnull().sum().sum()} | test: {Xtest.isnull().sum().sum()}")

# ─────────────────────────────────────────────────────────────────────────────
# 3.  FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
print("\n  STEP 3: Feature engineering")

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Log-transform heavy-tailed high-signal features
    df["X35_log"]       = np.log1p(df["X35"].clip(lower=0))
    df["X34_log"]       = np.log1p(df["X34"].clip(lower=0))

    # Ratio features (correlated with defect zones)
    df["ratio_35_36"]   = df["X35"] / (df["X36"].abs() + 1)
    df["ratio_34_37"]   = df["X34"] / (df["X37"].abs() + 1)
    df["ratio_35_37"]   = df["X35"] / (df["X37"].abs() + 1)

    # Aggregate of top-4 differentiating features
    df["sum_top4"]      = df[["X35", "X34", "X36", "X37"]].sum(axis=1)
    df["max_top4"]      = df[["X35", "X34", "X36", "X37"]].max(axis=1)

    # Stage-level temperature statistics
    temp_cols_early  = ["X1","X2","X3"]
    temp_cols_mid    = ["X4","X5","X6","X7","X8","X9"]
    temp_cols_late   = ["X17","X18","X19","X20"]
    all_temp_cols    = temp_cols_early + temp_cols_mid + temp_cols_late

    df["stage1_mean"]   = df[temp_cols_early].mean(axis=1)
    df["stage2_mean"]   = df[temp_cols_mid].mean(axis=1)
    df["stage3_mean"]   = df[temp_cols_late].mean(axis=1)
    df["temp_drop_1_2"] = df["stage1_mean"] - df["stage2_mean"]
    df["temp_drop_2_3"] = df["stage2_mean"] - df["stage3_mean"]
    df["temp_range"]    = df[all_temp_cols].max(axis=1) - df[all_temp_cols].min(axis=1)
    df["temp_std"]      = df[all_temp_cols].std(axis=1)

    # Cross-feature interactions
    df["prod_X2_X35log"] = df["X2"]  * df["X35_log"]
    df["diff_X13_X19"]   = df["X13"] - df["X19"]
    df["X38_X39"]        = df["X38"] * df["X39"]
    df["X40_X41"]        = df["X40"] * df["X41"]

    # Roughness / vibration proxy
    df["param_var"]     = df[FEATURE_COLS].var(axis=1)

    return df

X     = add_features(X)
Xtest = add_features(Xtest)
print(f"  Features after engineering: {X.shape[1]}")

# ─────────────────────────────────────────────────────────────────────────────
# 4.  SCALING
# ─────────────────────────────────────────────────────────────────────────────
print("\n  STEP 4: Robust scaling")
scaler  = RobustScaler()
Xs      = pd.DataFrame(scaler.fit_transform(X),      columns=X.columns)
Xtest_s = pd.DataFrame(scaler.transform(Xtest),       columns=X.columns)

# ─────────────────────────────────────────────────────────────────────────────
# 5.  ANOMALY SCORE (Isolation Forest — adds unsupervised signal)
# ─────────────────────────────────────────────────────────────────────────────
print("\n  STEP 5: Adding Isolation Forest anomaly score")
iso = IsolationForest(n_estimators=500, contamination=0.05,
                      max_samples="auto", random_state=42, n_jobs=-1)
iso.fit(Xs)
Xs["anomaly_score"]      = -iso.score_samples(Xs)
Xtest_s["anomaly_score"] = -iso.score_samples(Xtest_s)

COLS = list(Xs.columns)
print(f"  Total features (incl. anomaly): {len(COLS)}")

# ─────────────────────────────────────────────────────────────────────────────
# 6.  MODEL DEFINITIONS
#     Metric focus: 100% Recall, maximise Precision
#     Strategy   : High scale_pos_weight + threshold sweep post-training
# ─────────────────────────────────────────────────────────────────────────────
def make_xgb():
    return xgb.XGBClassifier(
        n_estimators     = 1000,
        max_depth        = 6,
        learning_rate    = 0.02,
        scale_pos_weight = 22,       # ~1286/66 ≈ 19.5; slightly over-weight
        subsample        = 0.80,
        colsample_bytree = 0.80,
        min_child_weight = 1,
        gamma            = 0.10,
        reg_alpha        = 0.05,
        reg_lambda       = 1.0,
        tree_method      = "hist",
        eval_metric      = "aucpr",
        use_label_encoder= False,
        random_state     = 42,
    )

def make_lgb():
    return lgb.LGBMClassifier(
        n_estimators     = 1000,
        max_depth        = 6,
        num_leaves       = 63,
        learning_rate    = 0.02,
        scale_pos_weight = 22,
        subsample        = 0.80,
        colsample_bytree = 0.80,
        min_child_samples= 5,
        reg_alpha        = 0.05,
        random_state     = 42,
        verbose          = -1,
    )

def make_cat():
    return CatBoostClassifier(
        iterations       = 600,
        depth            = 6,
        learning_rate    = 0.025,
        scale_pos_weight = 22,
        l2_leaf_reg      = 3,
        random_seed      = 42,
        verbose          = 0,
        train_dir        = CAT_DIR,
    )

def make_rf():
    return RandomForestClassifier(
        n_estimators   = 500,
        max_depth      = 12,
        min_samples_leaf= 2,
        class_weight   = {0: 1, 1: 22},
        random_state   = 42,
        n_jobs         = -1,
    )

# Ensemble weights (tuned empirically)
WEIGHTS = {"xgb": 0.35, "lgb": 0.30, "cat": 0.25, "rf": 0.10}

# ─────────────────────────────────────────────────────────────────────────────
# 7.  CROSS-VALIDATED TRAINING (10-fold stratified)
# ─────────────────────────────────────────────────────────────────────────────
print("\n  STEP 6: 10-Fold Stratified Cross-Validation + Ensemble")
print("  (SMOTETomek oversampling applied inside each fold to avoid leakage)\n")

N_FOLDS = 10
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
sm      = SMOTETomek(random_state=42)

oof_probas        = np.zeros(len(Xs))
test_probas_folds = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(Xs, y)):
    Xtr, Xval   = Xs.iloc[tr_idx][COLS].values,  Xs.iloc[val_idx][COLS].values
    ytr, yval_f = y[tr_idx],                      y[val_idx]

    # ── Oversample only on training split ──
    Xtr_sm, ytr_sm = sm.fit_resample(Xtr, ytr)

    # ── Fit each model ──
    xgb_m = make_xgb()
    lgb_m = make_lgb()
    cat_m = make_cat()
    rf_m  = make_rf()

    xgb_m.fit(Xtr_sm, ytr_sm,
              eval_set=[(Xval, yval_f)], verbose=False)

    lgb_m.fit(Xtr_sm, ytr_sm,
              eval_set=[(Xval, yval_f)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])

    cat_m.fit(Xtr_sm, ytr_sm,
              eval_set=(Xval, yval_f),
              early_stopping_rounds=50, verbose=0)

    rf_m.fit(Xtr_sm, ytr_sm)

    # ── Ensemble proba ──
    p = (WEIGHTS["xgb"] * xgb_m.predict_proba(Xval)[:, 1] +
         WEIGHTS["lgb"] * lgb_m.predict_proba(Xval)[:, 1] +
         WEIGHTS["cat"] * cat_m.predict_proba(Xval)[:, 1] +
         WEIGHTS["rf"]  * rf_m.predict_proba(Xval)[:, 1])

    oof_probas[val_idx] = p

    # ── Test fold predictions ──
    Xt = Xtest_s[COLS].values
    tp = (WEIGHTS["xgb"] * xgb_m.predict_proba(Xt)[:, 1] +
          WEIGHTS["lgb"] * lgb_m.predict_proba(Xt)[:, 1] +
          WEIGHTS["cat"] * cat_m.predict_proba(Xt)[:, 1] +
          WEIGHTS["rf"]  * rf_m.predict_proba(Xt)[:, 1])
    test_probas_folds.append(tp)

    # ── Quick fold report ──
    pred_at_10 = (p >= 0.10).astype(int)
    r = recall_score(yval_f, pred_at_10)
    p_ = precision_score(yval_f, pred_at_10, zero_division=0)
    auc = roc_auc_score(yval_f, p)
    print(f"  Fold {fold+1:2d}: Recall={r:.3f}  Precision={p_:.3f}  AUC-ROC={auc:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# 8.  THRESHOLD SELECTION (on OOF — zero leakage)
#     Rule: Find smallest threshold that gives Recall=1.0, then pick the one
#           with highest Precision (= least false positives).
# ─────────────────────────────────────────────────────────────────────────────
print("\n  STEP 7: Threshold selection on OOF predictions")

best_thresh = 0.05
best_prec   = 0.0

thresh_candidates = np.arange(0.005, 0.70, 0.001)
for t in thresh_candidates:
    pred = (oof_probas >= t).astype(int)
    r  = recall_score(y, pred, zero_division=0)
    p_ = precision_score(y, pred, zero_division=0)
    if r == 1.0 and p_ > best_prec:
        best_prec   = p_
        best_thresh = t

pred_oof = (oof_probas >= best_thresh).astype(int)
oof_recall = recall_score(y, pred_oof)
oof_prec   = precision_score(y, pred_oof, zero_division=0)
oof_f1     = f1_score(y, pred_oof)

print(f"\n  ✅ Best threshold : {best_thresh:.4f}")
print(f"  OOF Recall      : {oof_recall:.4f}  (target = 1.000)")
print(f"  OOF Precision   : {oof_prec:.4f}  (target > 0.900)")
print(f"  OOF F1          : {oof_f1:.4f}")
print()
print("  Full OOF classification report:")
print(classification_report(y, pred_oof, target_names=["No Defect","Defect"]))

# If Recall < 1.0 or Precision < 0.5, warn the user
if oof_recall < 1.0:
    print("  ⚠️  WARNING: OOF Recall < 1.0 — threshold may need manual lowering.")
if oof_prec < 0.50:
    print("  ⚠️  NOTE: Precision is below 50% on OOF. Consider Optuna tuning (see below).")

# ─────────────────────────────────────────────────────────────────────────────
# 9.  FINAL TEST PREDICTIONS
# ─────────────────────────────────────────────────────────────────────────────
print("\n  STEP 8: Generating final test predictions")

test_probas_final = np.mean(test_probas_folds, axis=0)
pred_test = (test_probas_final >= best_thresh).astype(int)

print(f"  Defects predicted in test set : {pred_test.sum()} / {len(pred_test)}")

# ─────────────────────────────────────────────────────────────────────────────
# 10.  SAVE SUBMISSION
# ─────────────────────────────────────────────────────────────────────────────
submission = pd.DataFrame({
    "CoilID": test["CoilID"].values,
    "Y"     : pred_test,
})

out_path = os.path.join(OUT_DIR, "submission.csv")
submission.to_csv(out_path, index=False)

print(f"\n  ✅ Submission saved → {out_path}")
print(f"  Shape: {submission.shape}")
print(f"  Prediction distribution:\n{submission['Y'].value_counts().to_string()}")

# ─────────────────────────────────────────────────────────────────────────────
# 11.  OPTIONAL: Probability threshold report (to manually pick if needed)
# ─────────────────────────────────────────────────────────────────────────────
print("\n  STEP 9 [Optional]: Threshold sensitivity table (OOF)")
print(f"  {'Thresh':>8}  {'Recall':>8}  {'Precision':>10}  {'Test Defects':>13}")
print("  " + "-"*48)
for t in np.arange(0.01, 0.30, 0.01):
    pred = (oof_probas >= t).astype(int)
    r    = recall_score(y, pred, zero_division=0)
    p_   = precision_score(y, pred, zero_division=0)
    test_n = (test_probas_final >= t).sum()
    flag = " ◀ chosen" if abs(t - best_thresh) < 0.005 else ""
    print(f"  {t:>8.3f}  {r:>8.3f}  {p_:>10.3f}  {test_n:>13}{flag}")

print("""
=============================================================================
  DONE!  Upload submission.csv to HackerEarth.
=============================================================================
""")

  STEP 1: Loading data
  Train : (1352, 51)  |  Defects: 66 / 1352  (4.9%)
  Test  : (339, 50)

  STEP 2: Imputing missing values
  Missing after impute — train: 0 | test: 0

  STEP 3: Feature engineering
  Features after engineering: 68

  STEP 4: Robust scaling

  STEP 5: Adding Isolation Forest anomaly score
  Total features (incl. anomaly): 69

  STEP 6: 10-Fold Stratified Cross-Validation + Ensemble
  (SMOTETomek oversampling applied inside each fold to avoid leakage)

  Fold  1: Recall=1.000  Precision=0.194  AUC-ROC=0.9790
  Fold  2: Recall=1.000  Precision=0.132  AUC-ROC=0.9358
  Fold  3: Recall=1.000  Precision=0.045  AUC-ROC=0.8359
  Fold  4: Recall=1.000  Precision=0.107  AUC-ROC=0.8049
  Fold  5: Recall=1.000  Precision=0.065  AUC-ROC=0.7829
  Fold  6: Recall=1.000  Precision=0.044  AUC-ROC=0.7158
  Fold  7: Recall=1.000  Precision=0.135  AUC-ROC=0.9040
  Fold  8: Recall=0.857  Precision=0.128  AUC-ROC=0.8694
  Fold  9: Recall=0.857  Precision=0.115  AUC-ROC=0.8493
  Fold 1